# EVASPA tiling 

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path

In [ ]:
if os.environ["EVASPA_TEST_DATA_PATH"] is None:
    raise Exception("Variable EVASPA_TEST_DATA_PATH must be set")
if os.environ["MNT_PATH"] is None:
    raise Exception("Variable MNT_PATH must be set")

In [ ]:
import evaspa.api as api
import evaspa.tiling as tiling
import evaspa.trishna as trishna

In [ ]:
outdir = "../report"
os.makedirs(outdir,exist_ok=True)

### Read ROI and generate list of tiles

In [ ]:
roi_tiles, adjs = api.generate_tiles(Path("roi_italy.shp"),land_percentage=5)

In [ ]:
print(f"Number of tiles: {len(roi_tiles)}")

In [ ]:
roi = gpd.read_file("roi_italy.shp")
roi = roi.set_crs(4326)
fig = plt.figure(figsize=(8, 8))
ax = plt.gca()
ax = roi.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                   )
cx.add_basemap(ax,crs=roi.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("ROI")
fig.savefig(os.path.join(outdir,"roi.png"))

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.gca()
ax = roi_tiles.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                   )
cx.add_basemap(ax,crs=roi.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("ROI")
fig.savefig(os.path.join(outdir,"roi_tiles.png"))

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.gca()
ax = roi_tiles.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                    column="land", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Percentage of land per tile")
fig.savefig(os.path.join(outdir,"roi_tiles_landpercentage.png"))

### Init

In [ ]:
land = trishna.get_land_mask()

In [ ]:
tile_df, group_df = tiling.initialize_regroup(roi_tiles, land)

In [ ]:
group_df[["nbvalid"]].reset_index()

In [ ]:
roi_tiles["utm"] = roi_tiles.apply(lambda row: row.id[0:2], axis=1)

In [ ]:
gdf = roi_tiles[['id','geometry','utm']].merge(group_df[["pvalid","nbvalid"]].reset_index(), how='inner', left_on="id",right_on="group").drop(columns="group")

In [ ]:
gdf["pvalid"] *=100

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.gca()
ax = gdf.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                    column="pvalid", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Percentage of valid pixels per tile")
fig.savefig(os.path.join(outdir,"roi_tiles_pvalid.png"))

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.gca()
ax = gdf.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                    column="nbvalid", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Number of valid pixels per tile")
fig.savefig(os.path.join(outdir,"roi_tiles_nbvalid.png"))

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=1, figsize=(24, 8),
                            layout="constrained")
# Tiles
ax = gdf.plot(ax=axs[0], 
                    alpha=0.8, 
                    edgecolor="k",
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("ROI Tiles")
# nbvalid
ax = gdf.plot(ax=axs[2], 
                    alpha=0.8, 
                    edgecolor="k",
                    column="nbvalid", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Number of valid pixels per tile")
# UTM
ax = gdf.plot(ax=axs[1], 
                    alpha=0.8, 
                    edgecolor="k",
                    column="utm", 
                    categorical=True,
                    cmap="Pastel1", 
                    legend=True,
                    legend_kwds={
                        "loc":"lower right",
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("UTM zones")
fig.savefig(os.path.join(outdir,"roi_tiles.png"))

### Regroup tiles

In [ ]:
group = api.regroup_tiles(roi_tiles, adjs, threshold=600000)

In [ ]:
group.head()

In [ ]:
tiling.write_regroup(group.reset_index(),os.path.join(outdir,"regroup.shp"))

In [ ]:
gdf2 = gpd.read_file(os.path.join(outdir,"regroup.shp"))

In [ ]:
gdf2.head()

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.gca()
ax = gdf2.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                   )
cx.add_basemap(ax,crs=roi.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("ROI")
fig.savefig(os.path.join(outdir,"regroup.png"))

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.gca()
ax = gdf2.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                    column="nbvalid", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Number of valid pixels per tile")
fig.savefig(os.path.join(outdir,"regroup_nbvalid.png"))

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(16, 8),
                            layout="constrained")
# Tiles
ax = gdf2.plot(ax=axs[0], 
                    alpha=0.8, 
                    edgecolor="k",
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Tile association")
# nbvalid
ax = gdf2.plot(ax=axs[1], 
                    alpha=0.8, 
                    edgecolor="k",
                    column="nbvalid", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Number of valid pixels per group")
fig.savefig(os.path.join(outdir,"regroup.png"))

In [ ]:
gdf2

In [ ]:
gdf

In [ ]:
out, _ = pd.cut(
    gdf["nbvalid"],
    bins=np.arange(0, 3700000, 600000),
    include_lowest=True,
    right=False,
    retbins=True,
)
count = out.value_counts()
print(count)
#print(count.index[0], count.values[0])
#print(count.index[1], count.values[1])
#print("rest", count.values[2:].sum())

In [ ]:
out, _ = pd.cut(
    gdf2["nbvalid"],
    bins=np.arange(0, 3700000, 600000),
    include_lowest=True,
    right=False,
    retbins=True,
)
count2 = out.value_counts()
print(count2)

In [ ]:
for i in count.index:
    print(i, count.loc[i],count2.loc[i])